In [2]:
pip install librosa

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 8.6 MB/s eta 0:00:01
   ------- -------------------------------- 0.5/2.8 MB 8.6 MB/s eta 0:00:01
   ------- -------------------------------- 0.5/2.8 MB 8.6 MB/s eta 0:00:01
   ------- -------------------------------- 0.5/2.8 MB 8.6 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 835.3 kB/s eta 0:00:03
   ---------------------------------- ----- 2.4/2.8 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 2.1 MB/s  0:00:01
   ---------------------------------------- 0.0/39.2 MB ? eta -:--:--
   - -------------------------------------- 1.0/39.2 MB 7.0 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/39.2 MB 6.5 MB/s eta 0:00:06
   --- ------------------------------------ 3.4/39.2 MB 6.4 MB/s eta 0:00:06
   ----- ---------------------------------- 5.5/39.2 MB 6.8 MB/s eta 0:00:05
   ------ ------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import librosa
import numpy as np

In [4]:
def load_audio(file_path):
    """
    Carga un archivo de audio y devuelve la señal y la frecuencia de muestreo.
    
    Args:
        file_path (str): La ruta al archivo de audio.

    Returns:
        y (np.ndarray): La señal de audio.
        sr (int): La frecuencia de muestreo.
    """ 
    
    
    y, sr = librosa.load(file_path, sr=None)
    return y, sr

def extract_features(y, sr):
    """
    Extrae features clave del audio.
    
    Returns:
        tempo (float)
        chroma_mean (array)
    """
    # 🎵 tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    
    # 🎹 chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = chroma.mean(axis=1)
    
    return tempo, chroma_mean
    

In [5]:
NOTE_MAP = ['C', 'C#', 'D', 'D#', 'E', 'F',
            'F#', 'G', 'G#', 'A', 'A#', 'B']

In [6]:
def detect_key(chroma_mean):
    """
    Detecta la nota dominante.
    
    Returns:
        key (str)
    """
    note_index = np.argmax(chroma_mean)
    key = NOTE_MAP[note_index]
    return key

In [7]:

def detect_key(chroma_mean):
    """
    Detecta la nota dominante.
    
    Returns:
        key (str)
    """
    weights = chroma_mean / np.sum(chroma_mean)
    note_index = np.argmax(weights)
    key = NOTE_MAP[note_index]
    return key
    

In [8]:
def detect_mode(tempo, chroma_mean):
    """
    Detecta modo basado en heurísticas simples.
    
    Returns:
        "major" o "minor"
    """
    energy = np.mean(chroma_mean)
    
    if tempo < 90 and energy < 0.5:
        return "minor"
    else:
        return "major"

In [9]:
def get_chords(key, mode):
    """
    Genera progresión de acordes básica.
    """
    if mode == "minor":
        return [f"{key}m", "F", "C", "G"]
    else:
        return [key, "G", "Am", "F"]

In [13]:
def analyze_audio(file_path):
    """
    Pipeline completo de análisis de audio.
    
    Returns:
        dict con resultados
    """
    y, sr = load_audio(file_path)
    
    tempo, chroma_mean = extract_features(y, sr)
    
    key = detect_key(chroma_mean)
    
    mode = detect_mode(tempo, chroma_mean)
    
    chords = get_chords(key, mode)
    
    return {
        "tempo": float(tempo[0]),
        "key": key,
        "mode": mode,
        "chords": chords
    }

In [14]:
result = analyze_audio("beat.wav")

print("🎧 Resultado:")
print(result)

🎧 Resultado:
{'tempo': 120.18531976744185, 'key': 'D', 'mode': 'major', 'chords': ['D', 'G', 'Am', 'F']}


In [15]:
result

{'tempo': 120.18531976744185,
 'key': 'D',
 'mode': 'major',
 'chords': ['D', 'G', 'Am', 'F']}